# Étape 1 — Data Acquisition & Prétraitement
**Projet :** Génération de données synthétiques de sinistres pour la tarification en assurance  
**Dataset :** Insurance Claims Data — 58 592 polices, 41 variables  
**Auteurs :** Groupe ISFA 2025-2026

---

## Contenu
1. Imports et configuration
2. Acquisition — Chargement des données brutes
3. Contrôle qualité
4. Préparation — Suppression de l'identifiant
5. Transformation — Parsing max_torque et max_power
6. Transformation — Encodage binaire Yes/No
7. Transformation — One-hot encoding
8. Déséquilibre des classes
9. Normalisation StandardScaler
10. Sauvegarde


## 1. Imports et configuration

On importe les librairies nécessaires :
- **pandas** : manipulation des tableaux de données
- **numpy** : calculs numériques
- **re** : expressions régulières pour le parsing de texte
- **os** : gestion des dossiers et fichiers
- **StandardScaler** : normalisation des données numériques

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports OK")

## 2. Acquisition — Chargement des données brutes

On charge le fichier CSV source avec `pandas.read_csv()`.

**Pourquoi `sep=','` ?** Le fichier utilise la virgule comme séparateur de colonnes.

On vérifie immédiatement deux choses :
- Le nombre de lignes et colonnes
- L'absence de valeurs manquantes

In [ ]:
df_raw = pd.read_csv('../data/Insurance claims data.csv', sep=',')
print(f"[1] Données brutes : {df_raw.shape[0]} lignes, {df_raw.shape[1]} colonnes")
print(f"    Valeurs manquantes : {df_raw.isnull().sum().sum()}")
print()
print("Aperçu des 3 premières lignes :")
display(df_raw.head(3))

## 3. Contrôle qualité

Avant toute transformation, on s'assure que les données sont fiables.

4 vérifications :
1. **Doublons** — des polices identiques en double ?
2. **Valeurs manquantes** — des cellules vides ?
3. **Plages de valeurs** — des valeurs impossibles (âge négatif, etc.) ?
4. **Outliers** — des valeurs extrêmes via la méthode IQR ?

**Méthode IQR :** un point est aberrant si il est en dehors de [Q1 - 1.5×IQR, Q3 + 1.5×IQR]

In [ ]:
print("[2] Contrôle qualité :")

# 2a. Doublons
n_doublons = df_raw.duplicated().sum()
print(f"    Doublons : {n_doublons}")
if n_doublons > 0:
    df_raw = df_raw.drop_duplicates()
    print(f"    → {n_doublons} doublon(s) supprimé(s)")

# 2b. Valeurs manquantes
n_missing   = df_raw.isnull().sum()
cols_missing = n_missing[n_missing > 0]
if len(cols_missing) == 0:
    print(f"    Valeurs manquantes : aucune ✓")
else:
    print(f"    Valeurs manquantes détectées :")
    print(cols_missing)

In [ ]:
# 2c. Plages de valeurs
print("    Plages de valeurs :")
print(f"      customer_age        : [{df_raw['customer_age'].min()} – {df_raw['customer_age'].max()}] ans")
print(f"      vehicle_age         : [{df_raw['vehicle_age'].min()} – {df_raw['vehicle_age'].max()}] ans")
print(f"      subscription_length : [{df_raw['subscription_length'].min()} – {df_raw['subscription_length'].max()}] ans")
print(f"      region_density      : [{df_raw['region_density'].min()} – {df_raw['region_density'].max()}]")
print(f"      airbags             : [{df_raw['airbags'].min()} – {df_raw['airbags'].max()}]")
print(f"      ncap_rating         : [{df_raw['ncap_rating'].min()} – {df_raw['ncap_rating'].max()}]")

# Assertions de cohérence métier
assert (df_raw['customer_age'] >= 18).all(), "Âge client < 18 détecté"
assert (df_raw['vehicle_age'] >= 0).all(),   "Âge véhicule négatif détecté"
assert (df_raw['airbags'] >= 0).all(),        "Nombre d'airbags négatif détecté"
assert df_raw['claim_status'].isin([0, 1]).all(), "claim_status contient des valeurs hors {0,1}"
print("    Vérifications de cohérence métier : OK ✓")

In [ ]:
# 2d. Outliers via IQR
print("    Outliers détectés (méthode IQR) :")
numeric_check = ['customer_age', 'vehicle_age', 'region_density', 'displacement', 'gross_weight']

for col in numeric_check:
    Q1   = df_raw[col].quantile(0.25)
    Q3   = df_raw[col].quantile(0.75)
    IQR  = Q3 - Q1
    mask = (df_raw[col] < Q1 - 1.5*IQR) | (df_raw[col] > Q3 + 1.5*IQR)
    n_out = mask.sum()
    print(f"      {col:<25} : {n_out} outliers ({n_out/len(df_raw)*100:.1f}%)")

print()
print("    → Outliers CONSERVÉS")
print("    Justification : en assurance, les valeurs extrêmes (vieux véhicules,")
print("    zones très denses) sont des cas réels à modéliser.")

## 4. Préparation — Suppression de l'identifiant

`policy_id` est un identifiant unique par police — il ne contient **aucune information** sur le risque.

Le conserver poserait deux problèmes :
- Le modèle apprendrait à mémoriser des polices spécifiques (surapprentissage)
- Problème de confidentialité des données

In [ ]:
df = df_raw.drop(columns=['policy_id'])
print(f"[3] Suppression de policy_id → {df.shape[1]} colonnes restantes")

## 5. Transformation — Parsing max_torque et max_power

Ces deux colonnes arrivent sous forme de texte brut qu'un modèle ne peut pas lire :
- `max_torque` : format **"250Nm@2750rpm"**
- `max_power` : format **"113.45bhp@4000rpm"**

On utilise des **expressions régulières (regex)** pour extraire les valeurs numériques cachées dans ces chaînes.

On crée 4 nouvelles colonnes numériques :
- `torque_nm` — couple en Newton-mètres
- `torque_rpm` — régime du couple
- `power_bhp` — puissance en chevaux
- `power_rpm` — régime de la puissance

In [ ]:
def parse_torque(s):
    m = re.search(r'([\d.]+)Nm', str(s))
    return float(m.group(1)) if m else np.nan

def parse_torque_rpm(s):
    m = re.search(r'@([\d.]+)rpm', str(s))
    return float(m.group(1)) if m else np.nan

def parse_power(s):
    m = re.search(r'([\d.]+)bhp', str(s))
    return float(m.group(1)) if m else np.nan

def parse_power_rpm(s):
    m = re.search(r'@([\d.]+)rpm', str(s))
    return float(m.group(1)) if m else np.nan

# Test sur un exemple
exemple = "250Nm@2750rpm"
print(f"Exemple : '{exemple}'")
print(f"  torque_nm  = {parse_torque(exemple)}")
print(f"  torque_rpm = {parse_torque_rpm(exemple)}")

In [ ]:
# Application sur tout le dataset
df['torque_nm']  = df['max_torque'].apply(parse_torque)
df['torque_rpm'] = df['max_torque'].apply(parse_torque_rpm)
df['power_bhp']  = df['max_power'].apply(parse_power)
df['power_rpm']  = df['max_power'].apply(parse_power_rpm)
df = df.drop(columns=['max_torque', 'max_power'])

print(f"[4] Parsing max_torque/max_power → 4 colonnes numériques extraites")
print(f"    NaN après parsing : torque_nm={df['torque_nm'].isnull().sum()}, power_bhp={df['power_bhp'].isnull().sum()}")
print()
print("Aperçu des nouvelles colonnes :")
display(df[['torque_nm','torque_rpm','power_bhp','power_rpm']].head(3))

## 6. Transformation — Encodage binaire Yes/No → 0/1

17 colonnes représentent des équipements de sécurité sous forme textuelle `"Yes"` / `"No"`.

Un modèle ML ne peut pas traiter du texte — on les convertit en entiers **0** et **1**.

Exemples de colonnes concernées : `is_esc`, `is_tpms`, `is_parking_camera`, etc.

In [ ]:
binary_cols = [
    'is_esc', 'is_adjustable_steering', 'is_tpms', 'is_parking_sensors',
    'is_parking_camera', 'is_front_fog_lights', 'is_rear_window_wiper',
    'is_rear_window_washer', 'is_rear_window_defogger', 'is_brake_assist',
    'is_power_door_locks', 'is_central_locking', 'is_power_steering',
    'is_driver_seat_height_adjustable', 'is_day_night_rear_view_mirror',
    'is_ecw', 'is_speed_alert'
]

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

print(f"[5] Encodage binaire Yes/No sur {len(binary_cols)} colonnes")
print()
print("Vérification — valeurs uniques après encodage :")
for col in binary_cols[:3]:
    print(f"  {col} : {sorted(df[col].unique())}") 

## 7. Transformation — One-hot encoding des variables catégorielles

8 colonnes sont des variables catégorielles sans ordre naturel :
`fuel_type`, `segment`, `transmission_type`, `rear_brakes_type`, `steering_type`, `engine_type`, `model`, `region_code`

**Pourquoi pas des entiers (0, 1, 2...) ?**  
Encoder "Diesel=0, Petrol=1, CNG=2" créerait un ordre fictif — le modèle croirait que CNG > Petrol > Diesel, ce qui n'a aucun sens.

**Le one-hot encoding** crée une colonne binaire par modalité.  
Par exemple `fuel_type` devient 3 colonnes : `fuel_type_Diesel`, `fuel_type_Petrol`, `fuel_type_CNG`.

In [ ]:
nominal_cols = ['fuel_type', 'transmission_type', 'rear_brakes_type',
                'steering_type', 'segment', 'engine_type', 'model', 'region_code']

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=False)

print(f"[6] One-hot encoding sur {len(nominal_cols)} colonnes catégorielles")
print(f"    Avant  : {df.shape[1]} colonnes")
print(f"    Après  : {df_encoded.shape[1]} colonnes")
print(f"    Gain   : +{df_encoded.shape[1] - df.shape[1]} colonnes binaires créées")

## 8. Déséquilibre des classes

C'est l'observation centrale qui **justifie tout le projet**.

Si seulement 6,4% des polices ont un sinistre, un modèle naïf qui dit toujours "pas de sinistre" aurait 93,6% de précision — mais il serait totalement inutile en assurance car il ne détecterait aucun accident.

In [ ]:
import matplotlib.pyplot as plt

n_claims = df_encoded['claim_status'].sum()
n_total  = len(df_encoded)
ratio    = n_claims / n_total * 100

print(f"[7] Déséquilibre des classes :")
print(f"    Sinistres (1)     : {int(n_claims):>6} ({ratio:.1f}%)")
print(f"    Non-sinistres (0) : {n_total - int(n_claims):>6} ({100-ratio:.1f}%)")
print(f"    Ratio d'imbalance : 1 pour {int((n_total - n_claims) / n_claims)}")

# Visualisation
fig, ax = plt.subplots(figsize=(6, 5))
counts = df_encoded['claim_status'].value_counts()
ax.pie(counts, labels=['Non-sinistre (93.6%)', 'Sinistre (6.4%)'],
       colors=['#2E86AB', '#E74C3C'], autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title("Déséquilibre des classes — claim_status", fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Normalisation — StandardScaler

Les 16 colonnes numériques ont des échelles très différentes :
- `customer_age` : entre 35 et 75
- `region_density` : jusqu'à 73 430
- `displacement` : en cm³

Sans normalisation, les grandes valeurs écrasent les petites dans l'apprentissage.

**StandardScaler** transforme chaque colonne pour avoir :
- Moyenne = 0
- Écart-type = 1

Formule : **x' = (x - μ) / σ**

⚠️ `claim_status` n'est PAS normalisé — il reste en 0/1.

In [ ]:
X = df_encoded.drop(columns=['claim_status'])
y = df_encoded['claim_status']

numeric_cols = [
    'subscription_length', 'vehicle_age', 'customer_age', 'region_density',
    'displacement', 'cylinder', 'turning_radius', 'length', 'width',
    'gross_weight', 'torque_nm', 'torque_rpm', 'power_bhp', 'power_rpm',
    'airbags', 'ncap_rating'
]

scaler   = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print(f"[8] Normalisation (StandardScaler) sur {len(numeric_cols)} colonnes numériques")
print()
print("Contrôle post-normalisation (mean ≈ 0, std ≈ 1) :")
for col in numeric_cols[:5]:
    print(f"  {col:<25} → mean={X_scaled[col].mean():.4f}, std={X_scaled[col].std():.4f}")

## 10. Sauvegarde

On produit **deux fichiers distincts** pour des usages différents dans les étapes suivantes :

| Fichier | Contenu | Usage |
|---------|---------|-------|
| `data_encoded.csv` | Données en échelle naturelle | EDA (étape 2) + CTGAN/TVAE (étape 3) |
| `data_preprocessed.csv` | Données normalisées | Classifieur XGBoost (étapes 3-4) |

**Pourquoi deux fichiers ?**  
CTGAN et TVAE préfèrent les données en échelle naturelle pour apprendre les vraies distributions.  
XGBoost fonctionne mieux avec des données normalisées.

In [ ]:
os.makedirs('../outputs', exist_ok=True)

X_scaled['claim_status'] = y.values
X_scaled.to_csv('../outputs/data_preprocessed.csv', index=False)
df_encoded.to_csv('../outputs/data_encoded.csv', index=False)

print(f"[9] Fichiers sauvegardés dans outputs/")
print(f"    data_preprocessed.csv ({X_scaled.shape[0]:,} x {X_scaled.shape[1]})")
print(f"    data_encoded.csv       ({df_encoded.shape[0]:,} x {df_encoded.shape[1]})")
print()
print("✓ Étape 1 terminée.")

## Résumé — Étape 1

In [ ]:
print("=" * 55)
print("  RÉSUMÉ ÉTAPE 1 — PRÉTRAITEMENT")
print("=" * 55)
print(f"  Données brutes    : 58 592 × 41")
print(f"  Données finales   : {df_encoded.shape[0]:,} × {df_encoded.shape[1]}")
print()
print("  Transformations effectuées :")
print("    ✓ Contrôle qualité (doublons, manquants, outliers)")
print("    ✓ Suppression policy_id")
print("    ✓ Parsing max_torque / max_power → 4 colonnes")
print("    ✓ Encodage Yes/No → 0/1 (17 colonnes)")
print("    ✓ One-hot encoding (8 catégorielles → ~53 colonnes)")
print("    ✓ Normalisation StandardScaler (16 colonnes)")
print()
print(f"  Déséquilibre : 3 748 sinistres / 54 844 non-sinistres")
print(f"  Ratio        : 1 pour 14")
print()
print("  Fichiers produits :")
print("    → outputs/data_encoded.csv")
print("    → outputs/data_preprocessed.csv")